In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

housing = fetch_california_housing(as_frame=True)
df = housing.frame
features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms"]
X = df[features]
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [2]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
linear_r2 = r2_score(y_test, linear_pred)

In [3]:
lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1, max_iter=10000)
)
lasso_model.fit(X_train, y_train)
lasso_pred = lasso_model.predict(X_test)
lasso_r2 = r2_score(y_test, lasso_pred)

lasso_coef = lasso_model.named_steps["lasso"].coef_
lasso_zero = int(np.sum(np.isclose(lasso_coef, 0.0)))
lasso_nonzero = int(np.sum(~np.isclose(lasso_coef, 0.0)))
linear_nonzero = int(np.sum(~np.isclose(linear_model.coef_, 0.0)))

results = pd.DataFrame({
    "Модель": ["LinearRegression", "Lasso (alpha=0.1)"],
    "Test R²": [linear_r2, lasso_r2],
    "Нульові коеф.": [0, lasso_zero],
    "Ненульові коеф.": [linear_nonzero, lasso_nonzero]
})

print(results.to_string(index=False, formatters={"Test R²": "{:.4f}".format}))
print(f"Кількість коефіцієнтів Lasso, що стали нульовими: {lasso_zero}")


           Модель Test R²  Нульові коеф.  Ненульові коеф.
 LinearRegression  0.5089              0                4
Lasso (alpha=0.1)  0.4797              2                2
Кількість коефіцієнтів Lasso, що стали нульовими: 2
